### Imports

In [1]:
import sys
sys.path.insert(0, '/Users/chengyouyu/Desktop/Gastrulation/final_stuff/models')
sys.path.insert(0, '/Users/chengyouyu/Desktop/Gastrulation/final_stuff/visualizers')
from model_mechanics import *
import pickle
import matplotlib.pyplot as plt
import seaborn
import visuals as asys
import numpy as np


### Running the simulation

### Updated Simulation Parameters

In [57]:
N = 430               # Number of cells in the simulation
N0    = 240               # Fraction of type 0 cells if using make_random_sphere() cell generation function
R = (N-N0)**(1/3)+1            # Radius of initial cell distribution
seed = 42

np.random.seed(seed)

with open('inits/data_full_42.npy', 'rb') as f:
    mask_lst, x_lst, p_lst, q_lst = pickle.load(f)

x, p, q, m = x_lst[-1], p_lst[-1], q_lst[-1], mask_lst[-1]

# Specify DVE cells

    # Select 11 cells (Takaoka et al. 2011) with the lowest z-coordinates
ve_idx = np.where(m == 1)[0]
DVE_cells = ve_idx[np.argsort(x[ve_idx, 2])[:11]]
m[DVE_cells] = 2

# Select only half of the embryo
bottom = x[:, 2] < 0
x, p, q, m = x[bottom], p[bottom], q[bottom], m[bottom]

init_conditions = (m, x, p, q)

frames = 1000
perframe = 100
t_full = frames * perframe

# Run simulation with these initial conditions
# Simulation parameters
sim_dict = {
    # Data and output
    'output_folder'     : "inits",                                         # Output folder for simulation data
    'data'              : init_conditions,   # The double ellipse
    'yield_every'       : perframe,                                               # How often the simulation yields data. 
    'yield_steps'       : frames,                                             # How many data-yields we want. Total number of timesteps is yield_every * yield_steps

    # Stuff for tensors
    'device'            : 'cpu',           # Device to run the simulation on. 'cuda' or 'cpu'
    'dtype'             : torch.float,      # Data type for tensors. Either float32 or float64

    # Proliferation parameters
    'prolif_rate'       : 0,           # Cell division probabilities
    'prolif_delay'      : 0,                # Timesteps before the cells begin proliferating
    'max_cells'         : 10_000,           # Maximum number of cells in the simulation. When this number is reached the simulation terminates

    # Simulation parameters
    'dt'                : 0.01,              # Time step for the simulation
    'eta'               : 0.01,            # Noise level for the simulation
    'lambdas'           : [[0, 0.7, 0.0, 0.0],  #0-0 (ePI-epI)
                        [0.0, 0.8, 0.0, 0.0],     #1-1 (emVE-emVE)
                        [0.0, 0.6, 0.2, 0.0],      #2-2 (dve-dve)
                        [0.2, 0.0, 0.0, 0.0],      #0-1 (epi-emVE)
                        [0.2, 0.0, 0.0, 0.0],     #0-2 (epi-dve)
                        [0.0, 0.6, 0.0, 0.0]],    #1-2 (emVE-dve)
    'offsets'           : [-0.5,                   #0-0
                            -1.6,                   #1-1
                            -1.6,                   #2-2
                            -1,                   #0-1
                            -1,                   #0-2
                            -1.6],                  #1-2    

    # 'lambdas'           : [0.4, 0.3, 0, 0],
        
    'egg_shape'         : [R+3,R+3,2*R+6],             # If set, we enact boundary conditions on a spherical region of this radius
    'z_wall_k'          : 10, 
    'ceiling_z'         : 0,
    'push'              : 0.06,           #Active force for cellular motion    




    # Miscellaneous
    'notes'             : f'Running test w. bounds',    # Notes about the simulation. Will be printed when the simulation starts if verbose and will also be saved in .json output
    'verbose'           : True,                         # Verbosity level for the simulation. Set off when debugging
    'random_seed'       : seed,                           # Random seed for the simulation
    'init_number'      : f'_cut_{seed}',                            # To keep track of which initial condition we are using

    }

run_simulation(sim_dict=sim_dict)           # Let's run the simulation

Using input data
Starting simulation with notes:
Running test w. bounds
Simulation done, saved 1000 datapoints00)   (210 cells)
Took 300.44134283065796 seconds


In [58]:
file = "inits/data_cut_42.npy"
asys.animate(file, if_U = False, morph = -1, cam_angle=(90,0,0), frame_t = 50)
